In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Project 4 - Databricks Streaming Gold Layer
# MAGIC
# MAGIC Builds the gold-layer output for the project's second locked streaming use case:
# MAGIC real-time driving-behavior anomaly detection from telematics (the first, claims
# MAGIC fraud/risk scoring at intake, runs off the batch claim_events leg instead). Two
# MAGIC tables:
# MAGIC
# MAGIC - `gold_telematics_anomalies` - event-level, one row per silver reading, flagged
# MAGIC   `is_anomaly` when either the per-PID statistical outlier flag or a high
# MAGIC   alarm_class (>= 3) fires. Feeds a detail/drill-down view.
# MAGIC - `gold_device_risk_summary` - device-level rollup (event count, anomaly count/
# MAGIC   rate, worst alarm_class seen, most recent event time). Feeds the "Real-Time
# MAGIC   Fraud & Telematics" dashboard page's per-device risk view directly.
# MAGIC
# MAGIC Full-snapshot build, same as the batch gold layer's dims (`nb_gold_databricks.py`) -
# MAGIC no SCD2 here either, since a device's risk summary is inherently a point-in-time
# MAGIC recomputation over the current event window, not a dimension with a change history
# MAGIC to version.
# MAGIC
# MAGIC Run after `nb_streaming_silver_databricks` (needs `silver_telematics_stream`).

# COMMAND ----------

from pyspark.sql import DataFrame, functions as F
from datetime import datetime, timezone


def log_pipeline_run(spark, platform, layer, start_dt, end_dt):
    duration = round((end_dt - start_dt).total_seconds(), 2)
    log_row = spark.createDataFrame([{
        "platform": platform, "layer": layer,
        "start_ts": start_dt, "end_ts": end_dt, "duration_seconds": duration,
    }])
    log_row.write.format("delta").mode("append").saveAsTable("pipeline_run_log")
    print(f"[{platform}/{layer}] duration: {duration}s")

# COMMAND ----------

# alarm_class >= HIGH_ALARM_THRESHOLD counts as anomalous on its own, independent of
# the statistical VALUE_OUTLIER flag - the documented schema treats 0-4 as an
# escalating severity scale, so a high class is a domain-level anomaly signal even
# when the raw value isn't a statistical outlier relative to that PID's own history
HIGH_ALARM_THRESHOLD = 3


def build_gold_telematics_anomalies(silver_df: DataFrame) -> DataFrame:
    df = silver_df.withColumn(
        "is_anomaly",
        F.col("VALUE_OUTLIER") | (F.coalesce(F.col("alarm_class"), F.lit(0)) >= HIGH_ALARM_THRESHOLD)
    )
    df = df.withColumn("event_date", F.to_date(F.from_unixtime(F.col("timestamp") / 1000)))
    return df.select("device_id", "timestamp", "event_date", "PID", "value", "alarm_class", "VALUE_OUTLIER", "is_anomaly")


def build_gold_device_risk_summary(anomalies_df: DataFrame) -> DataFrame:
    return anomalies_df.groupBy("device_id").agg(
        F.count("*").alias("total_events"),
        F.sum(F.col("is_anomaly").cast("int")).alias("anomaly_count"),
        F.round(F.avg(F.col("is_anomaly").cast("int")), 4).alias("anomaly_rate"),
        F.max(F.coalesce(F.col("alarm_class"), F.lit(0))).alias("max_alarm_class"),
        F.max("timestamp").alias("last_event_timestamp"),
    )

# COMMAND ----------

start_dt = datetime.now(timezone.utc)

silver_stream = spark.read.table("silver_telematics_stream")

gold_anomalies_df = build_gold_telematics_anomalies(silver_stream)
gold_anomalies_df.write.format("delta").mode("overwrite").saveAsTable("gold_telematics_anomalies")

gold_risk_summary_df = build_gold_device_risk_summary(gold_anomalies_df)
gold_risk_summary_df.write.format("delta").mode("overwrite").saveAsTable("gold_device_risk_summary")

end_dt = datetime.now(timezone.utc)
log_pipeline_run(spark, "Databricks", "streaming_gold", start_dt, end_dt)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Sanity check
# MAGIC `gold_telematics_anomalies` row count should equal `silver_telematics_stream`'s
# MAGIC exactly (event-level, no join, no row loss - same shape as batch's
# MAGIC `fact_telematics`). `gold_device_risk_summary` row count should equal the distinct
# MAGIC device_id count from the producer notebook (10, by default). `anomaly_rate` should
# MAGIC be a small fraction, not close to 0 or 1 - either extreme suggests
# MAGIC `HIGH_ALARM_THRESHOLD` or the upstream outlier flag needs a second look before
# MAGIC trusting this for the dashboard.

# COMMAND ----------

gold_anomalies_check = spark.read.table("gold_telematics_anomalies")
gold_risk_summary_check = spark.read.table("gold_device_risk_summary")

print("gold_telematics_anomalies rows:", gold_anomalies_check.count())
print("gold_device_risk_summary rows (should equal distinct device count):", gold_risk_summary_check.count())
print()
display(gold_risk_summary_check.orderBy(F.desc("anomaly_rate")))